# Preprocessing BiLSTM untuk Prediksi Sebaran Abu (+2 Jam)

Notebook ini menyiapkan data untuk BiLSTM multi-output.
Target: `jarak_km`, `luas_km2`, `radius_km`, dan arah (`sudut`) dalam representasi `sin/cos`.

Catatan:
- `timestamp` dipakai sebagai fitur turunan siklik (`hour/month/day_of_year`).
- `id` dibuang.
- artifacts preprocessing disimpan ke folder artifacts.
- durasi pipeline diukur dengan `time.perf_counter()`.

In [19]:
from pathlib import Path
import json
import pickle
import time

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [20]:
# Konfigurasi path
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "java-ash-hysplit.csv"
PROCESSED_DIR = BASE_DIR / "processed_bilstm_2h"
ARTIFACT_DIR = BASE_DIR / "artifacts_bilstm_preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_PATH exists:", DATA_PATH.exists())
print("PROCESSED_DIR:", PROCESSED_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

BASE_DIR: d:\Projects\volcanic_ash\modeling
DATA_PATH exists: True
PROCESSED_DIR: d:\Projects\volcanic_ash\modeling\processed_bilstm_2h
ARTIFACT_DIR: d:\Projects\volcanic_ash\modeling\artifacts_bilstm_preprocessing


In [21]:
# Mulai perf counter untuk seluruh pipeline preprocessing
pipeline_start = time.perf_counter()

In [22]:
# Load dan validasi kolom
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

target_cols_raw = ["jarak_km", "luas_km2", "sudut_deg", "radius_km"]
required_cols = [
    "timestamp", "volcano_filter", "alert_level",
    "latitude", "longitude", "elevation",
    "tinggi_letusan_m", "kec_angin_km_jam", "arah_angin_deg",
    "amplitudo", "duration"
] + target_cols_raw

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["timestamp"] + target_cols_raw).copy()
df = df.sort_values(["volcano_filter", "timestamp"]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
display(df.head(3))

Cleaned shape: (1707, 16)


,id,timestamp,volcano_filter,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km
0,1621,2016-04-02 04:19:00+00:00,Bromo,Yellow,-7.941944,112.95,2329,9285.0,4.6,51.0,8.0,124.0,47.387776,398.761300,241.819555,23.880052
1,1620,2016-04-02 04:45:00+00:00,Bromo,Yellow,-7.941944,112.95,2329,9605.0,4.6,51.0,9.0,124.0,51.962811,460.109193,246.342065,26.582662
2,1619,2016-04-03 04:07:00+00:00,Bromo,Yellow,-7.941944,112.95,2329,8645.0,6.1,45.0,9.0,124.0,27.527365,245.406718,218.947322,13.949546


In [23]:
# Feature engineering
df["arah_angin_rad"] = np.deg2rad(df["arah_angin_deg"] % 360)
df["wind_dir_sin"] = np.sin(df["arah_angin_rad"])
df["wind_dir_cos"] = np.cos(df["arah_angin_rad"])

df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.month
df["day_of_year"] = df["timestamp"].dt.dayofyear

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

# Target sudut direpresentasikan sin/cos agar cocok untuk regresi kontinu
df["target_sudut_sin"] = np.sin(np.deg2rad(df["sudut_deg"] % 360))
df["target_sudut_cos"] = np.cos(np.deg2rad(df["sudut_deg"] % 360))

lag_features = ["amplitudo", "duration", "kec_angin_km_jam", "tinggi_letusan_m"]
for col in lag_features:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("volcano_filter", dropna=False)[col].shift(lag)

df = df.dropna().reset_index(drop=True)
print("Shape after FE + lags:", df.shape)

Shape after FE + lags: (1694, 42)


In [24]:
# Encoding + pemilihan fitur/target
df_model = pd.get_dummies(df, columns=["volcano_filter", "alert_level"], drop_first=False)
df_model["__time_order__"] = df["timestamp"].values

drop_cols = ["id", "timestamp", "arah_angin_deg", "arah_angin_rad", "sudut_deg"]
drop_cols = [c for c in drop_cols if c in df_model.columns]
df_model = df_model.drop(columns=drop_cols)

target_final = ["jarak_km", "luas_km2", "radius_km", "target_sudut_sin", "target_sudut_cos"]
feature_cols = [c for c in df_model.columns if c not in target_final + ["__time_order__"]]

print("Jumlah fitur:", len(feature_cols))
print("Target final:", target_final)

Jumlah fitur: 38
Target final: ['jarak_km', 'luas_km2', 'radius_km', 'target_sudut_sin', 'target_sudut_cos']


In [25]:
# Time-based split
df_model = df_model.sort_values("__time_order__").reset_index(drop=True)

n = len(df_model)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_final].copy()
y_val = val_df[target_final].copy()
y_test = test_df[target_final].copy()

print("Split shapes (X):", X_train.shape, X_val.shape, X_test.shape)
print("Split shapes (y):", y_train.shape, y_val.shape, y_test.shape)

Split shapes (X): (1185, 38) (254, 38) (255, 38)
Split shapes (y): (1185, 5) (254, 5) (255, 5)


In [26]:
# Scaling dan sequence building
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

def make_sequences(X_df, y_df, lookback=6):
    X_vals = X_df.values
    y_vals = y_df.values
    X_seq, y_seq = [], []
    for i in range(lookback, len(X_vals)):
        X_seq.append(X_vals[i - lookback:i])
        y_seq.append(y_vals[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

LOOKBACK = 6
X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train, lookback=LOOKBACK)
X_val_seq, y_val_seq = make_sequences(X_val_scaled, y_val, lookback=LOOKBACK)
X_test_seq, y_test_seq = make_sequences(X_test_scaled, y_test, lookback=LOOKBACK)

print("Sequence shape train:", X_train_seq.shape, y_train_seq.shape)
print("Sequence shape val:", X_val_seq.shape, y_val_seq.shape)
print("Sequence shape test:", X_test_seq.shape, y_test_seq.shape)

Sequence shape train: (1179, 6, 38) (1179, 5)
Sequence shape val: (248, 6, 38) (248, 5)
Sequence shape test: (249, 6, 38) (249, 5)


In [27]:
# Simpan output preprocess + artifacts
train_out = pd.concat([X_train_scaled, y_train], axis=1)
val_out = pd.concat([X_val_scaled, y_val], axis=1)
test_out = pd.concat([X_test_scaled, y_test], axis=1)

train_out.to_csv(PROCESSED_DIR / "train_preprocessed.csv", index=False)
val_out.to_csv(PROCESSED_DIR / "val_preprocessed.csv", index=False)
test_out.to_csv(PROCESSED_DIR / "test_preprocessed.csv", index=False)

np.save(PROCESSED_DIR / "X_train_seq.npy", X_train_seq)
np.save(PROCESSED_DIR / "y_train_seq.npy", y_train_seq)
np.save(PROCESSED_DIR / "X_val_seq.npy", X_val_seq)
np.save(PROCESSED_DIR / "y_val_seq.npy", y_val_seq)
np.save(PROCESSED_DIR / "X_test_seq.npy", X_test_seq)
np.save(PROCESSED_DIR / "y_test_seq.npy", y_test_seq)

with open(ARTIFACT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(ARTIFACT_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with open(ARTIFACT_DIR / "target_cols.json", "w", encoding="utf-8") as f:
    json.dump(target_final, f, indent=2)

elapsed_sec = time.perf_counter() - pipeline_start
meta = {
    "lookback": LOOKBACK,
    "n_features": len(feature_cols),
    "targets": target_final,
    "train_seq_shape": list(X_train_seq.shape),
    "val_seq_shape": list(X_val_seq.shape),
    "test_seq_shape": list(X_test_seq.shape),
    "preprocessing_seconds": float(elapsed_sec)
}

with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Selesai preprocessing dalam detik:", round(elapsed_sec, 3))
print("Saved processed files in:", PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.glob("*")):
    print("-", p.name)

print("Saved artifacts in:", ARTIFACT_DIR)
for p in sorted(ARTIFACT_DIR.glob("*")):
    print("-", p.name)

Selesai preprocessing dalam detik: 0.301
Saved processed files in: d:\Projects\volcanic_ash\modeling\processed_bilstm_2h
- test_preprocessed.csv
- train_preprocessed.csv
- val_preprocessed.csv
- X_test_seq.npy
- X_train_seq.npy
- X_val_seq.npy
- y_test_seq.npy
- y_train_seq.npy
- y_val_seq.npy
Saved artifacts in: d:\Projects\volcanic_ash\modeling\artifacts_bilstm_preprocessing
- feature_cols.json
- metadata.json
- scaler.pkl
- target_cols.json
